In [72]:
import pandas as pd
import pickle

# Removed the custom neuralnet imports because Scikit-Learn handles it now!

# 1. THE CORRECT ALPHABETICAL MAP (From your LabelEncoder training)
REVERSE_CLASS_MAP = {
    0: 'Human', 
    1: 'OpenAI', 
    2: 'Google', 
    3: 'Meta', 
    4: 'Anthropic'
}

# 2. Load the test dataset
df_teste = pd.read_csv('subm1.csv', sep=';')

# CRITICAL: Do NOT lemmatize or remove punctuation! 
# The character-level TF-IDF needs the raw text exactly as it is.
textos_teste = df_teste['Text'].fillna('')

# 3. Load Vectorizer and Model
with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

with open('melhor_modelo_svc.pkl', 'rb') as f:
    modelo_svc = pickle.load(f) # Renamed to reflect that it's an SVC, not your Numpy DNN

# 4. Transform and Predict
# Pro-tip: Scikit-learn models can handle the sparse matrix directly, 
# so we can drop the .toarray() to save a massive amount of RAM!
X_teste_tabular = vectorizer.transform(textos_teste)

# Use .predict() instead of .forward(). 
# It directly outputs the [0, 1, 2, 3, 4] indices, so no need for np.argmax!
previsoes_idx = modelo_svc.predict(X_teste_tabular)

# 5. Map back to strings using the correct dictionary
df_teste['Label'] = [REVERSE_CLASS_MAP[idx] for idx in previsoes_idx]
nome_ficheiro_saida = 'subm1-g6-MIA-A.csv'
df_teste.to_csv(nome_ficheiro_saida, index=False, sep=';')

print(f"Sucesso! Ficheiro {nome_ficheiro_saida} gerado.")
print("\nNova Distribuição de Previsões:")
print(df_teste['Label'].value_counts())

Sucesso! Ficheiro subm1-g6-MIA-A.csv gerado.

Nova Distribuição de Previsões:
Label
Human        39
Meta         27
Google       20
Anthropic    12
OpenAI        2
Name: count, dtype: int64


C:\Users\bruno\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\bruno\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\brun

In [73]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# 1. Load the ground truth and your predictions
df_real = pd.read_csv('subm1_labels_revealed.csv', sep=';')
df_subA = pd.read_csv('subm1-g6-MIA-A.csv', sep=';')

# (If you want to test B, just change the file name above!)

# 2. Merge them together using the ID column to ensure they line up perfectly
merged = pd.merge(df_subA, df_real, on='ID', suffixes=('_pred', '_real'))

# 3. Calculate the final accuracy score
accuracy = accuracy_score(merged['Label_real'], merged['Label_pred'])
print(f"✨ Final Accuracy Score: {accuracy * 100:.2f}% ✨\n")

# 4. Print the detailed breakdown to see the "Google Bias" in action
print("Detailed Breakdown (Precision & Recall):")
print(classification_report(merged['Label_real'], merged['Label_pred']))

✨ Final Accuracy Score: 72.00% ✨

Detailed Breakdown (Precision & Recall):
              precision    recall  f1-score   support

   Anthropic       0.83      0.59      0.69        17
      Google       0.75      0.88      0.81        17
       Human       0.77      0.88      0.82        34
        Meta       0.63      0.94      0.76        18
      OpenAI       0.00      0.00      0.00        14

    accuracy                           0.72       100
   macro avg       0.60      0.66      0.62       100
weighted avg       0.64      0.72      0.67       100

